# 🚗 Mitsubishi AI Knowledge Assistant — RAG Pipeline
### Gemini + Pinecone + LangChain + RAGAS Evaluation

| Phase | Topik |
|-------|-------|
| **Setup** | Install & konfigurasi Gemini + Pinecone |
| **Phase 01** | Data Ingestion — DirectoryLoader, Chunking |
| **Phase 02** | Embedding & Vector Database — Pinecone |
| **Phase 03** | Hybrid Search — Dense + BM25 + RRF |
| **Phase 04** | RAG Pipeline — LangChain LCEL |
| **Phase 05** | Q&A — 5 Pertanyaan Mitsubishi |
| **Phase 06** | RAGAS Evaluation — Faithfulness, Relevance, Precision |

> **Struktur folder yang diharapkan:**
> ```
> data/
> ├── xpander_spec.txt
> ├── xpander_review.txt
> ├── pajero_sport_spec.txt
> └── ... (file .txt lainnya)
> ```

## ⚙️ Setup — Install & Konfigurasi

In [ ]:
# ============================================================
# CELL 1: Install semua package yang dibutuhkan
# ============================================================
%pip install -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    langchain-pinecone \
    google-generativeai \
    pinecone-client \
    pinecone-text \
    python-dotenv \
    tiktoken \
    rank-bm25

In [ ]:
# ============================================================
# CELL 2: Konfigurasi API Keys
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv()

# Isi langsung di sini ATAU simpan di file .env
GOOGLE_API_KEY   = os.getenv("GOOGLE_API_KEY",   "your-google-api-key-here")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "your-pinecone-api-key-here")

# Nama index Pinecone — buat dulu di dashboard pinecone.io
# Dimension: 768 (gemini-embedding-004), Metric: dotproduct (untuk hybrid)
PINECONE_INDEX_NAME = "mitsubishi-rag"

os.environ["GOOGLE_API_KEY"]   = GOOGLE_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

print("✅ Konfigurasi loaded!")
print(f"   Google API Key : {'*' * 10}{GOOGLE_API_KEY[-4:]}")
print(f"   Pinecone Key   : {'*' * 10}{PINECONE_API_KEY[-4:]}")
print(f"   Pinecone Index : {PINECONE_INDEX_NAME}")

In [ ]:
# ============================================================
# CELL 3: Inisialisasi LLM & Embedding Model
# ============================================================
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# LLM untuk generate jawaban
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_tokens=1024,
)

# Embedding model untuk mengubah teks ke vektor
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-004",  # 768 dimensi
)

# Quick test
test_response = llm.invoke("Sebutkan satu model Mitsubishi populer di Indonesia.")
print("✅ LLM aktif:", test_response.content)

test_embed = embeddings.embed_query("Mitsubishi Xpander")
print(f"✅ Embedding aktif: dimensi = {len(test_embed)}")

## 📥 Phase 01 — Data Ingestion & Chunking

In [ ]:
# ============================================================
# CELL 4: Load semua dokumen dari folder data/
# ============================================================
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import os

DATA_DIR = "./data"  # <-- Ganti dengan path folder dokumenmu

loader = DirectoryLoader(
    path=DATA_DIR,
    glob="**/*.txt",           # load semua .txt termasuk subfolder
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
    use_multithreading=True,
)

raw_docs = loader.load()
print(f"\n✅ Total dokumen ter-load: {len(raw_docs)}")
for doc in raw_docs[:5]:
    print(f"   - {doc.metadata['source']} ({len(doc.page_content)} chars)")

In [ ]:
# ============================================================
# CELL 5: Metadata Enrichment dari nama file
# ============================================================
# Konvensi nama file yang diharapkan:
#   xpander_spec.txt      → model=xpander, doc_type=spec
#   pajero_sport_review.txt → model=pajero_sport, doc_type=review

for doc in raw_docs:
    filename = os.path.basename(doc.metadata["source"]).lower()
    name_clean = filename.replace(".txt", "")

    # Deteksi tipe dokumen
    if "spec" in name_clean:
        doc.metadata["doc_type"] = "spec"
        model_name = name_clean.replace("_spec", "").replace("spec_", "")
    elif "review" in name_clean:
        doc.metadata["doc_type"] = "review"
        model_name = name_clean.replace("_review", "").replace("review_", "")
    else:
        doc.metadata["doc_type"] = "general"
        model_name = name_clean

    doc.metadata["model_name"] = model_name.strip("_")
    doc.metadata["brand"]      = "mitsubishi"

print("✅ Contoh metadata setelah enrichment:")
for doc in raw_docs[:3]:
    print(f"   {doc.metadata}")

In [ ]:
# ============================================================
# CELL 6: Chunking dokumen
# ============================================================
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,           # optimal untuk dokumen spec/review pendek
    chunk_overlap=100,        # overlap agar konteks antar chunk tidak putus
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len,
)

chunks = splitter.split_documents(raw_docs)

print(f"✅ Total chunks: {len(chunks)}")
print(f"   Dari {len(raw_docs)} dokumen → rata-rata {len(chunks)//len(raw_docs)} chunk/dokumen")
print(f"\n   Sample chunk ke-1:")
print(f"   Metadata : {chunks[0].metadata}")
print(f"   Content  : {chunks[0].page_content[:200]}...")

## 🧮 Phase 02 — Embedding & Pinecone Vector Store

In [ ]:
# ============================================================
# CELL 7: Inisialisasi Pinecone Index
# ============================================================
# PENTING: Buat index terlebih dahulu di https://app.pinecone.io
# Settings index:
#   - Name       : mitsubishi-rag
#   - Dimensions : 768  (sesuai gemini-embedding-004)
#   - Metric     : dotproduct  (wajib untuk hybrid search)
#   - Type       : Dense (untuk hybrid, pilih "sparse-dense")

from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

# Cek index yang tersedia
existing_indexes = [idx.name for idx in pc.list_indexes()]
print(f"✅ Index yang tersedia: {existing_indexes}")

if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"⚠️  Index '{PINECONE_INDEX_NAME}' belum ada.")
    print("   Buat manual di https://app.pinecone.io dengan:")
    print("   - Dimensions: 768")
    print("   - Metric    : dotproduct")
else:
    index = pc.Index(PINECONE_INDEX_NAME)
    print(f"✅ Terhubung ke index '{PINECONE_INDEX_NAME}'")
    print(f"   Stats: {index.describe_index_stats()}")

In [ ]:
# ============================================================
# CELL 8: Embed & Upload ke Pinecone via LangChain
# ============================================================
from langchain_pinecone import PineconeVectorStore
import time

BATCH_SIZE = 50  # Pinecone free tier: max 100 per batch

print(f"📤 Mengupload {len(chunks)} chunks ke Pinecone...")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Total batch: {(len(chunks) // BATCH_SIZE) + 1}")

# Upload batch pertama dan buat vectorstore
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks[:BATCH_SIZE],
    embedding=embeddings,
    index_name=PINECONE_INDEX_NAME,
)

# Upload sisa batch
for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    vectorstore.add_documents(batch)
    print(f"   ✅ Batch {i//BATCH_SIZE + 1}: {len(batch)} chunks di-upload")
    time.sleep(1)  # hindari rate limit

print(f"\n🎉 Selesai! Total: {len(chunks)} chunks tersimpan di Pinecone.")

In [ ]:
# ============================================================
# CELL 9: Verifikasi — Test Similarity Search
# ============================================================

# Load vectorstore yang sudah ada (tidak perlu upload ulang)
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
)

test_query = "spesifikasi mesin Mitsubishi Xpander"
results = vectorstore.similarity_search_with_score(test_query, k=3)

print(f"🔎 Query: '{test_query}'")
print(f"\nTop {len(results)} hasil:")
for i, (doc, score) in enumerate(results):
    print(f"\n  [{i+1}] Score: {score:.4f}")
    print(f"       Model : {doc.metadata.get('model_name', 'unknown')}")
    print(f"       Type  : {doc.metadata.get('doc_type', 'unknown')}")
    print(f"       Content: {doc.page_content[:120]}...")

## 🔍 Phase 03 — Hybrid Search (Dense + BM25 + RRF)

In [ ]:
# ============================================================
# CELL 10: Fungsi-fungsi Hybrid Search
# ============================================================
from typing import List, Tuple, Dict
from rank_bm25 import BM25Okapi

def dense_search(query: str, k: int = 10) -> List[Tuple[any, float]]:
    """Semantic search menggunakan vector embedding."""
    return vectorstore.similarity_search_with_score(query, k=k)

def bm25_search(query: str, all_chunks: List, k: int = 10) -> List[Tuple[int, float]]:
    """Keyword search menggunakan BM25."""
    tokenized_corpus = [doc.page_content.lower().split() for doc in all_chunks]
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(idx, scores[idx]) for idx in top_indices]

def reciprocal_rank_fusion(
    dense_results: List[Tuple],
    bm25_results: List[Tuple],
    all_chunks: List,
    k: int = 60,
    top_n: int = 5
) -> List[any]:
    """Gabungkan hasil dense + BM25 dengan formula RRF."""
    rrf_scores: Dict[str, float] = {}
    doc_map: Dict[str, any] = {}

    # Dense search ranking
    for rank, (doc, score) in enumerate(dense_results, start=1):
        doc_id = doc.page_content[:50]  # pakai awal konten sebagai ID unik
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)
        doc_map[doc_id] = doc

    # BM25 ranking
    for rank, (idx, score) in enumerate(bm25_results, start=1):
        doc = all_chunks[idx]
        doc_id = doc.page_content[:50]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k + rank)
        doc_map[doc_id] = doc

    # Sort by RRF score
    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [doc_map[doc_id] for doc_id in sorted_ids[:top_n]]


def hybrid_search(query: str, all_chunks: List, top_n: int = 5) -> List[any]:
    """Main hybrid search: dense + BM25 → RRF re-ranking."""
    dense_results = dense_search(query, k=10)
    bm25_results  = bm25_search(query, all_chunks, k=10)
    reranked_docs = reciprocal_rank_fusion(dense_results, bm25_results, all_chunks, top_n=top_n)
    return reranked_docs

print("✅ Fungsi Hybrid Search siap digunakan")

In [ ]:
# ============================================================
# CELL 11: Test Hybrid Search
# ============================================================

test_query = "kapasitas mesin dan torsi Xpander"
hybrid_results = hybrid_search(test_query, chunks, top_n=5)

print(f"🔎 Hybrid Search — Query: '{test_query}'")
print(f"   {len(hybrid_results)} dokumen terpilih setelah RRF re-ranking\n")

for i, doc in enumerate(hybrid_results):
    print(f"  [{i+1}] Model : {doc.metadata.get('model_name', '?')} "
          f"| Type: {doc.metadata.get('doc_type', '?')}")
    print(f"       Content: {doc.page_content[:150]}...")
    print()

## 🤖 Phase 04 — RAG Pipeline (LangChain LCEL)

In [ ]:
# ============================================================
# CELL 12: Bangun RAG Chain dengan LangChain LCEL
# ============================================================
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

# ── Prompt Template ──
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """Kamu adalah asisten AI yang ahli tentang mobil-mobil Mitsubishi.
Jawab pertanyaan pengguna HANYA berdasarkan konteks dokumen yang diberikan.
Jika informasi tidak ada di konteks, katakan 'Informasi tersebut tidak tersedia dalam dokumen saya.'
Berikan jawaban yang jelas, akurat, dan mudah dipahami dalam Bahasa Indonesia.
Sertakan detail teknis jika relevan."""),
    ("human",
     """Konteks dokumen:
{context}

Pertanyaan: {question}""")
])

# ── Helper: format dokumen menjadi string konteks ──
def format_docs(docs: List) -> str:
    formatted = []
    for i, doc in enumerate(docs):
        model = doc.metadata.get('model_name', 'unknown').upper()
        dtype = doc.metadata.get('doc_type', 'general')
        formatted.append(f"[Sumber {i+1} — {model} {dtype}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# ── Retriever dengan Hybrid Search ──
def hybrid_retriever(query: str) -> List:
    return hybrid_search(query, chunks, top_n=5)

# ── RAG Chain (LCEL) ──
rag_chain = (
    RunnableParallel(
        context  = RunnableLambda(hybrid_retriever) | RunnableLambda(format_docs),
        question = RunnablePassthrough()
    )
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# ── RAG Chain dengan output lengkap (termasuk sumber) ──
def rag_with_sources(query: str) -> Dict:
    """Jalankan RAG dan kembalikan jawaban + dokumen sumber."""
    retrieved_docs = hybrid_retriever(query)
    context = format_docs(retrieved_docs)

    answer = (
        RAG_PROMPT
        | llm
        | StrOutputParser()
    ).invoke({"context": context, "question": query})

    return {
        "query": query,
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "context": context,
    }

print("✅ RAG Chain siap!")
print("   Komponen: Hybrid Retriever → Format Docs → Prompt → Gemini → Output")

In [ ]:
# ============================================================
# CELL 13: Test RAG Chain
# ============================================================

test_q = "Apa saja fitur utama Mitsubishi Xpander?"
result = rag_with_sources(test_q)

print(f"❓ Pertanyaan: {result['query']}")
print(f"\n💡 Jawaban:\n{result['answer']}")
print(f"\n📚 Dokumen sumber ({len(result['retrieved_docs'])} chunk):")
for doc in result['retrieved_docs']:
    print(f"   - {doc.metadata.get('model_name')} [{doc.metadata.get('doc_type')}]")

## ❓ Phase 05 — 5 Pertanyaan tentang Mitsubishi

In [ ]:
# ============================================================
# CELL 14: Jawab 5 Pertanyaan tentang Dokumen Mitsubishi
# ============================================================

# Sesuaikan pertanyaan dengan konten dokumenmu
FIVE_QUESTIONS = [
    "Berapa kapasitas mesin dan tenaga yang dihasilkan Mitsubishi Xpander?",
    "Apa perbedaan fitur keselamatan antara Xpander dan Pajero Sport?",
    "Bagaimana ulasan pengguna tentang konsumsi bahan bakar Mitsubishi Xpander?",
    "Apa saja varian yang tersedia untuk Mitsubishi Pajero Sport dan perbedaannya?",
    "Berdasarkan review, apa kelebihan dan kekurangan utama Mitsubishi Xpander?",
]

qa_results = []

print("=" * 70)
print("  🚗 MITSUBISHI KNOWLEDGE ASSISTANT — 5 PERTANYAAN")
print("=" * 70)

for i, question in enumerate(FIVE_QUESTIONS, 1):
    print(f"\n❓ [{i}/5] {question}")
    print("-" * 60)

    result = rag_with_sources(question)
    qa_results.append(result)

    print(f"💡 {result['answer']}")
    print(f"\n📚 Sumber: ", end="")
    sources = [f"{d.metadata.get('model_name')}[{d.metadata.get('doc_type')}]"
               for d in result['retrieved_docs'][:3]]
    print(", ".join(set(sources)))
    print("=" * 70)

print(f"\n✅ Selesai menjawab {len(FIVE_QUESTIONS)} pertanyaan!")

## 📊 Phase 06 — RAGAS Evaluation

In [ ]:
# ============================================================
# CELL 15: Fungsi Evaluasi RAGAS (LLM-as-Judge)
# ============================================================
# Pendekatan: gunakan Gemini sendiri sebagai evaluator
# (tanpa library ragas eksternal — lebih fleksibel)
import json

def evaluate_faithfulness(context: str, answer: str) -> Dict:
    """RAGAS Faithfulness: apakah jawaban didukung penuh oleh konteks?"""
    messages = [
        ("system", """Kamu adalah evaluator RAG system.
Tugasmu: evaluasi apakah setiap klaim dalam JAWABAN didukung oleh KONTEKS.

Return HANYA JSON valid (tanpa markdown):
{"score": <0.0-1.0>, "supported_claims": <jumlah klaim didukung>,
 "total_claims": <total klaim>, "reasoning": "<penjelasan singkat>"}"""),
        ("human", f"KONTEKS:\n{context}\n\nJAWABAN:\n{answer}")
    ]
    prompt = ChatPromptTemplate.from_messages(messages)
    response = (prompt | llm | StrOutputParser()).invoke({})
    try:
        clean = response.strip().replace("```json", "").replace("```", "")
        return json.loads(clean)
    except Exception:
        return {"score": 0.0, "error": "parse failed", "raw": response[:200]}


def evaluate_answer_relevance(query: str, answer: str) -> Dict:
    """RAGAS Answer Relevance: seberapa relevan jawaban dengan pertanyaan?"""
    messages = [
        ("system", """Kamu adalah evaluator RAG system.
Tugasmu: evaluasi seberapa relevan JAWABAN terhadap PERTANYAAN.

Return HANYA JSON valid (tanpa markdown):
{"score": <0.0-1.0>, "is_complete": <true/false>,
 "is_on_topic": <true/false>, "reasoning": "<penjelasan singkat>"}"""),
        ("human", f"PERTANYAAN:\n{query}\n\nJAWABAN:\n{answer}")
    ]
    prompt = ChatPromptTemplate.from_messages(messages)
    response = (prompt | llm | StrOutputParser()).invoke({})
    try:
        clean = response.strip().replace("```json", "").replace("```", "")
        return json.loads(clean)
    except Exception:
        return {"score": 0.0, "error": "parse failed", "raw": response[:200]}


def evaluate_context_precision(query: str, retrieved_docs: List) -> Dict:
    """RAGAS Context Precision: seberapa presisi konteks yang di-retrieve?"""
    docs_text = "\n".join(
        [f"Chunk {i+1}: {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
    )
    messages = [
        ("system", """Kamu adalah evaluator RAG system.
Tugasmu: untuk setiap chunk yang di-retrieve, nilai apakah chunk tersebut
BENAR-BENAR DIPERLUKAN untuk menjawab pertanyaan.

Return HANYA JSON valid (tanpa markdown):
{"precision": <0.0-1.0>,
 "assessments": [{"chunk": 1, "relevant": true, "reason": "..."}]}"""),
        ("human", f"PERTANYAAN:\n{query}\n\nCHUNKS:\n{docs_text}")
    ]
    prompt = ChatPromptTemplate.from_messages(messages)
    response = (prompt | llm | StrOutputParser()).invoke({})
    try:
        clean = response.strip().replace("```json", "").replace("```", "")
        return json.loads(clean)
    except Exception:
        return {"precision": 0.0, "error": "parse failed", "raw": response[:200]}


def full_ragas_evaluation(query: str, top_k: int = 5) -> Dict:
    """Jalankan RAG + evaluasi ketiga metrik RAGAS sekaligus."""
    result = rag_with_sources(query)

    print(f"\n📝 Query   : {query}")
    print(f"💬 Answer  : {result['answer'][:200]}...")
    print(f"\n{'='*60}")
    print("📊 RAGAS EVALUATION")
    print(f"{'='*60}")

    faith = evaluate_faithfulness(result['context'], result['answer'])
    rel   = evaluate_answer_relevance(query, result['answer'])
    prec  = evaluate_context_precision(query, result['retrieved_docs'])

    f_score = faith.get('score', 0)
    r_score = rel.get('score', 0)
    p_score = prec.get('precision', 0)

    print(f"1️⃣  Faithfulness      : {f_score:.2f}  — {faith.get('reasoning', '')[:80]}")
    print(f"2️⃣  Answer Relevance  : {r_score:.2f}  — {rel.get('reasoning', '')[:80]}")
    print(f"3️⃣  Context Precision : {p_score:.2f}")
    print(f"{'='*60}")

    return {
        "query": query,
        "answer": result['answer'],
        "retrieved_docs": result['retrieved_docs'],
        "faithfulness": f_score,
        "answer_relevance": r_score,
        "context_precision": p_score,
    }

print("✅ Fungsi RAGAS evaluation siap!")

In [ ]:
# ============================================================
# CELL 16: Jalankan Evaluasi untuk Semua 5 Pertanyaan
# ============================================================
import time

eval_results = []

print("🚀 Memulai evaluasi RAGAS untuk 5 pertanyaan...\n")

for i, q in enumerate(FIVE_QUESTIONS, 1):
    print(f"\n[{i}/5] Evaluasi: {q[:60]}...")
    eval_result = full_ragas_evaluation(q)
    eval_results.append(eval_result)
    time.sleep(2)  # rate limit buffer

print("\n✅ Evaluasi selesai!")

In [ ]:
# ============================================================
# CELL 17: Tabel Ringkasan Hasil RAGAS
# ============================================================

print("\n" + "=" * 90)
print(f"{'RAGAS EVALUATION SUMMARY — MITSUBISHI RAG SYSTEM':^90}")
print("=" * 90)
print(f"  {'No':<4} {'Pertanyaan':<45} {'Faith':>7} {'Relev':>7} {'Prec':>7}")
print("-" * 90)

avg_faith = avg_rel = avg_prec = 0

for i, r in enumerate(eval_results, 1):
    q_short = r['query'][:43] + ".." if len(r['query']) > 43 else r['query']
    f = r['faithfulness']
    rv = r['answer_relevance']
    p = r['context_precision']
    avg_faith += f
    avg_rel   += rv
    avg_prec  += p
    print(f"  {i:<4} {q_short:<45} {f:>7.2f} {rv:>7.2f} {p:>7.2f}")

n = len(eval_results)
print("-" * 90)
print(f"  {'RATA-RATA':<49} {avg_faith/n:>7.2f} {avg_rel/n:>7.2f} {avg_prec/n:>7.2f}")
print("=" * 90)
print()
print("📌 Interpretasi skor (0.0 = buruk, 1.0 = sempurna):")
print("   Faithfulness      : Apakah jawaban didukung oleh konteks (anti-halusinasi)")
print("   Answer Relevance  : Apakah jawaban menjawab pertanyaan dengan tepat")
print("   Context Precision : Apakah dokumen yang di-retrieve memang relevan")

## 💬 Bonus — Interactive Q&A Session

In [ ]:
# ============================================================
# CELL 18: Interactive Q&A — Ganti pertanyaan dan jalankan ulang
# ============================================================

YOUR_QUESTION = "Bagaimana perbandingan harga dan fitur antara Xpander dan Xpander Cross?"

result = rag_with_sources(YOUR_QUESTION)

print(f"❓ Pertanyaan  : {result['query']}")
print(f"\n💡 Jawaban:\n{result['answer']}")
print(f"\n📚 Sumber dokumen yang digunakan:")
for i, doc in enumerate(result['retrieved_docs'], 1):
    m = doc.metadata
    print(f"   [{i}] {m.get('model_name','?').upper()} — {m.get('doc_type','?')} "
          f"| '{doc.page_content[:80]}...'")